## Prototype: 
### A 'first pass' to model that everything is working properly. Later to be refactored and tidied up into .py files.

Matt Shumway

In [102]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import trange
import random

torch.manual_seed(0)



In [119]:
class Gomoku():
    def __init__(self, **kwargs):
        """
        Initialize the game

        :param kwargs: keyword arguments
        """
        self.size = kwargs.get('size', 8)
        self.win_len = kwargs.get('win_len', 5)
        self.board = np.zeros((self.size, self.size), dtype=int)
        self.action_size = self.size ** 2

    def get_initial_state(self):
        """
        Get the initial state of the game
        
        :return: the initial state
        """
        return np.zeros((self.size, self.size), dtype=int)

    def get_next_state(self, state, action, player):
        """
        Step the game forward by one move

        :param state: the current state of the game
        :param action: the action to take
        :param player: the player taking the action

        :return: the new state of the game
        """
        if isinstance(action, int):
            action = (action // self.size, action % self.size)
        else:
            print(action)
        state[action] = player
        return state

    def get_valid_actions(self, state):
        """
        Get the valid actions for the current state
        
        :param state: the current state of the game
        
        :return: the valid actions
        """
        return np.where(state.flatten() == 0)[0].astype(np.uint8)

    def check_win(self, state, last_action):
        """
        Check if the game has been won

        :param state: the current state of the game
        :param last_action: the last action taken

        :return: True if the game has been won, False otherwise
        """
        if last_action is None:
            return False
        if isinstance(last_action, int):
            last_action = (last_action // self.size, last_action % self.size)

        player = state[last_action]
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]  # horizontal, vertical, diagonal, anti-diagonal
        for dx, dy in directions:
            if self.count_in_direction(state, last_action, player, dx, dy) + \
                self.count_in_direction(state, last_action, player, -dx, -dy) - 1 >= self.win_len:
                return True

        return False

    def count_in_direction(self, state, last_action, player, dx, dy):
        """
        Count the number of consecutive pieces in a direction
        
        :param state: the current state of the game
        :param last_action: the last action taken
        :param player: the player to count for
        :param dx: the x direction to count in
        :param dy: the y direction to count in
        
        :return: the number of consecutive pieces
        """
        count = 0
        x, y = last_action

        for step in range(self.win_len):
            r, c = x + step * dx, y + step * dy
            if 0 <= r < self.size and 0 <= c < self.size and state[r, c] == np.int64(player):
                count += 1
            else:
                break
        return count

    def get_value_and_done(self, state, last_action):
        """
        Get the value of the game and whether the game is done.
        win -> 1, draw -> 0, lose -> -1
        
        :param state: the current state of the game
        :param last_action: the last action taken
        :param player: the player to get the value for
        
        :return: the value of the game and whether the game is done
        """
        if self.check_win(state, last_action):
            return 1, True
        if np.all(state != 0):
            return 0, True
        return 0, False

    def get_opponent(self, player):
        """
        Get the opponent of the given player
        Players are 1 and -1

        :param player: the player
        
        :return: the opponent"""
        return -player

    def render(self, state):
        """
        Render the game state

        :param state: the current state of the game
        """
        print('   ' + ' '.join([f'{i:2d}' for i in range(self.size)]))
        for i, row in enumerate(state):
            print(f'{i:2d} ' + ' '.join([f'{x:2d}' for x in row]))

    def get_encoded_state(self, state):
        """
        Encode the state of the game
        
        :param state: the current state of the game
        
        :return: the encoded state
        """
        encoded_state = np.stack(
            (state == -1, state == 0, state == 1)
        ).astype(np.float32)

        if len(state.shape) == 3:
            encoded_state = np.swapaxes(encoded_state, 0, 1)
        
        return encoded_state

    def get_opponent_value(self, value):
        return -value
    
    def change_perspective(self, state, player):
        return state * player



In [86]:
# gomoku = Gomoku(game_size=5, win_len=3)
# player = 1

# state = gomoku.get_initial_state()

# while True:
#     gomoku.render(state)
#     valid_actions = gomoku.get_valid_actions(state)
#     pretty_valid_actions = [(a // gomoku.size, a % gomoku.size) for a in valid_actions]
#     print("valid actions: ", pretty_valid_actions)
#     action = input(f"Player {player} action: ")
#     if ',' in action:
#         action = tuple(map(int, action.split(',')))
#         action = action[0] * gomoku.size + action[1]
#     else:
#         action = int(action)

#     if action not in valid_actions:
#         print("Invalid action")
#         continue

#     state = gomoku.get_next_state(state, action, player)

#     if gomoku.check_win(state, action, player):
#         print(state)
#         print(f"Player {player} wins!")
#         break

#     player = -1 if player == 1 else 1



In [104]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),  # kernel_size=3, padding=1 keeps the size the same
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList([ResBlock(num_hidden) for _ in range(num_resBlocks)])

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.size * game.size, game.action_size)
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.size * game.size, 1),
            nn.Tanh()
        )
        self.to(device)

    def forward(self, x):
        x = self.startBlock(x)
        for block in self.backBone:
            x = block(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value


class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

In [105]:
gomoku = Gomoku(game_size=5, win_len=3)
state = gomoku.get_initial_state()
state = gomoku.get_next_state(state, 0, 1)
state = gomoku.get_next_state(state, 1, -1)

encoded_state = gomoku.get_encoded_state(state)
# print(encoded_state)

tensor_state = torch.tensor(encoded_state).unsqueeze(0)
# print(tensor_state.shape)

model = ResNet(gomoku, 4, 64)

policy, value = model(tensor_state)
value = value.item()
policy = torch.softmax(policy, dim=1).squeeze().detach().numpy()
print(value, policy, sep='\n')

plt.bar(range(len(policy)), policy)
plt.show()

TypeError: ResNet.__init__() missing 1 required positional argument: 'device'

In [109]:
class Node():
    def __init__(self, game, state, args, parent=None, last_action=None, prior=0, visit_count=0):
        self.game = game
        self.state = state
        self.args = args
        self.parent = parent
        self.last_action = last_action
        self.prior = prior

        self.children = []

        self.visit_count = visit_count
        self.value_sum = 0

    def is_expanded(self):
        return len(self.children) > 0
    
    def select_child(self):
        best_child = None
        best_ucb = -np.inf

        for child in self.children:
            ucb = self.get_ucb(child)
            if ucb > best_ucb:
                best_ucb = ucb
                best_child = child
        
        return best_child
    
    def get_ucb(self, child):
        if child.visit_count == 0:
            q_value = 0
        else:
            q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
        return q_value + self.args['cpuct'] * child.prior * np.sqrt(self.visit_count) / (1 + child.visit_count)

    def expand(self, policy):
        for action, prob in enumerate(policy):
            if prob > 0:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action, 1)
                child_state = self.game.change_perspective(child_state, -1)

                child = Node(self.game, child_state, self.args, parent=self, last_action=action, prior=prob)
                self.children.append(child)
        
        return child
    
    def backpropagate(self, value):
        self.value_sum += value
        self.visit_count += 1

        value = self.game.get_opponent_value(value)
        if self.parent is not None:
            self.parent.backpropagate(value)


class MCTS():
    def __init__(self, game, model, **kwargs):
        self.game = game
        self.num_simulations = kwargs.get('num_simulations', 100)
        self.model = model
        self.kwargs = kwargs

    @torch.no_grad()
    def search(self, state):
        root = Node(self.game, state, self.kwargs, visit_count=1)

        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(state), device=self.model.device).unsqueeze(0)
        )
        policy = torch.softmax(policy, dim=1).squeeze(0).cpu().numpy()

        policy = (1 - self.kwargs['dirichlet_eps']) * policy + \
            self.kwargs['dirichlet_eps'] * np.random.dirichlet(self.kwargs['dirichlet_alpha'] * np.ones(self.game.action_size)) 

        valid_actions = self.game.get_valid_actions(state)
        full_policy = np.zeros(self.game.action_size)
        full_policy[valid_actions] = policy[valid_actions]

        full_policy /= (full_policy.sum() + 1e-8)

        root.expand(full_policy)

        for _ in range(self.num_simulations):
            node = root

            while node.is_expanded():
                node = node.select_child()

            value, done = self.game.get_value_and_done(node.state, node.last_action)
            value = self.game.get_opponent_value(value)

            if not done:
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(node.state), device=self.model.device).unsqueeze(0)
                )
                policy = torch.softmax(policy, dim=1).squeeze(0).cpu().numpy()

                valid_actions = self.game.get_valid_actions(node.state)
                full_policy = np.zeros(self.game.action_size)
                full_policy[valid_actions] = policy[valid_actions]
                full_policy /= (full_policy.sum() + 1e-8)

                node = node.expand(full_policy)

            node.backpropagate(value)

        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.last_action] = child.visit_count
        action_probs /= action_probs.sum()

        return action_probs


            
            

In [112]:
class AlphaZero():
    def __init__(self, game, model, optimizer, **kwargs):
        self.game = game
        self.model = model
        self.optimizer = optimizer
        self.kwargs = kwargs
        self.mcts = MCTS(game, model, **kwargs)
    
    def train(self, memory):
        random.shuffle(memory)
        for batch_idx in range(0, len(memory), self.kwargs['batch_size']):
            sample = memory[batch_idx: min(len(memory) - 1, batch_idx + self.kwargs['batch_size'])]
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            state, policy_targets, value_targets = torch.tensor(state).float(), torch.tensor(policy_targets).float(), torch.tensor(value_targets).float()
            state, policy_targets, value_targets = state.to(self.model.device), policy_targets.to(self.model.device), value_targets.to(self.model.device)

            out_policy, out_value = self.model(state)
            loss_policy = F.cross_entropy(out_policy, policy_targets)
            loss_value = F.mse_loss(out_value, value_targets)
            loss = loss_policy + loss_value

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


    def self_play(self):
        memory = []
        player = 1
        state = self.game.get_initial_state()

        while True:
            neutral_state = self.game.change_perspective(state, player)
            action_probs = self.mcts.search(neutral_state)

            memory.append((neutral_state, action_probs, player))

            temperature_action_probs = action_probs ** (1 / self.kwargs['temperature'])
            temperature_action_probs /= temperature_action_probs.sum()
            action = np.random.choice(self.game.action_size, p=temperature_action_probs)

            state = self.game.get_next_state(state, action, player)

            value, done = self.game.get_value_and_done(state, action)

            if done:
                returnMemory = []
                for hist_neutral_state, hist_action_probs, hist_player in memory:
                    hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                    returnMemory.append((
                        self.game.get_encoded_state(hist_neutral_state),
                        hist_action_probs,
                        hist_outcome
                    ))
                return returnMemory

            player = self.game.get_opponent(player)

    def learn(self):
        for step in range(self.kwargs['num_iterations']):
            memory = []

            self.model.eval()
            for _ in trange(self.kwargs['num_self_play']):
                memory += self.self_play()
            
            self.model.train()
            for _ in trange(self.kwargs['num_training_steps']):
                self.train(memory)
            
            torch.save(self.model.state_dict(), f'model_{step}.pt')
            torch.save(self.optimizer.state_dict(), f'optimizer_{step}.pt')



In [128]:
class MCTSParallel():
    def __init__(self, game, model, **kwargs):
        self.game = game
        self.num_simulations = kwargs.get('num_simulations', 100)
        self.model = model
        self.kwargs = kwargs

    @torch.no_grad()
    def search(self, states, spGames):

        policy, _ = self.model(
            torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
        )
        policy = torch.softmax(policy, dim=1).cpu().numpy()

        policy = (1 - self.kwargs['dirichlet_eps']) * policy + \
            self.kwargs['dirichlet_eps'] * np.random.dirichlet(self.kwargs['dirichlet_alpha'] * np.ones(self.game.action_size), size=policy.shape[0]) 

        for i, spg in enumerate(spGames):
            spg_policy = policy[i]
            valid_actions = self.game.get_valid_actions(states[i])
            spg_full_policy = np.zeros(self.game.action_size)
            spg_full_policy[valid_actions] = spg_policy[valid_actions]

            spg_full_policy /= (spg_full_policy.sum() + 1e-8)

            spg.root = Node(self.game, states[i], self.kwargs, visit_count=1)

            spg.root.expand(spg_full_policy)

        for _ in range(self.num_simulations):
            for spg in spGames:
                spg.node = None
                node = spg.root

                while node.is_expanded():
                    node = node.select_child()

                value, done = self.game.get_value_and_done(node.state, node.last_action)
                value = self.game.get_opponent_value(value)

                if done:
                    node.backpropagate(value)
                else:
                    spg.node = node

            expandable_spGames = [mapping_idx for mapping_idx in range(len(spGames)) if spGames[mapping_idx].node is not None]

            if len(expandable_spGames) > 0:
                states = np.stack([spGames[mapping_idx].node.state for mapping_idx in expandable_spGames])
                policy, value = self.model(
                    torch.tensor(self.game.get_encoded_state(states), device=self.model.device)
                )
                policy = torch.softmax(policy, dim=1).cpu().numpy()
            
            for i, mapping_idx in enumerate(expandable_spGames):
                node = spGames[mapping_idx].node
                spg_policy, spg_value = policy[i], value[i]
                valid_actions = self.game.get_valid_actions(node.state)
                spg_full_policy = np.zeros(self.game.action_size)
                spg_full_policy[valid_actions] = spg_policy[valid_actions]
                spg_policy /= (spg_policy.sum() + 1e-8)

                node.expand(spg_full_policy)
                node.backpropagate(spg_value)


class AlphaZeroParallel():
    def __init__(self, game, model, optimizer, **kwargs):
        self.game = game
        self.model = model
        self.optimizer = optimizer
        self.kwargs = kwargs
        self.mcts = MCTSParallel(game, model, **kwargs)
    
    def train(self, memory):
        random.shuffle(memory)
        for batch_idx in range(0, len(memory), self.kwargs['batch_size']):
            sample = memory[batch_idx: min(len(memory) - 1, batch_idx + self.kwargs['batch_size'])]
            state, policy_targets, value_targets = zip(*sample)

            state, policy_targets, value_targets = np.array(state), np.array(policy_targets), np.array(value_targets).reshape(-1, 1)
            state, policy_targets, value_targets = torch.tensor(state).float(), torch.tensor(policy_targets).float(), torch.tensor(value_targets).float()
            state, policy_targets, value_targets = state.to(self.model.device), policy_targets.to(self.model.device), value_targets.to(self.model.device)

            out_policy, out_value = self.model(state)
            loss_policy = F.cross_entropy(out_policy, policy_targets)
            loss_value = F.mse_loss(out_value, value_targets)
            loss = loss_policy + loss_value

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()


    def self_play(self):
        return_memory = []
        player = 1
        spGames = [SPG(self.game) for _ in range(self.kwargs['num_parallel_games'])]
        
        while len(spGames) > 0:
            states = np.stack([spg.state for spg in spGames])
            neutral_states = self.game.change_perspective(states, player)

            self.mcts.search(neutral_states, spGames)

            for i in range(len(spGames))[::-1]:
                spg = spGames[i]

                action_probs = np.zeros(self.game.action_size)
                for child in spg.root.children:
                    action_probs[child.last_action] = child.visit_count
                action_probs /= action_probs.sum()

                spg.memory.append((spg.root.state, action_probs, player))

                temperature_action_probs = action_probs ** (1 / self.kwargs['temperature'])
                temperature_action_probs /= temperature_action_probs.sum()
                action = np.random.choice(self.game.action_size, p=temperature_action_probs)

                spg.state = self.game.get_next_state(spg.state, action, player)

                value, done = self.game.get_value_and_done(spg.state, action)

                if done:
                    for hist_neutral_state, hist_action_probs, hist_player in spg.memory:
                        hist_outcome = value if hist_player == player else self.game.get_opponent_value(value)
                        return_memory.append((
                            self.game.get_encoded_state(hist_neutral_state),
                            hist_action_probs,
                            hist_outcome
                        ))
                    del spGames[i]

            player = self.game.get_opponent(player)
        
        return return_memory

    def learn(self):
        for step in range(self.kwargs['num_iterations']):
            memory = []

            self.model.eval()
            for _ in trange(self.kwargs['num_self_play'] // self.kwargs['num_parallel_games']):
                memory += self.self_play()
            
            self.model.train()
            for _ in trange(self.kwargs['num_training_steps']):
                self.train(memory)
            
            torch.save(self.model.state_dict(), f'model_{step}.pt')
            torch.save(self.optimizer.state_dict(), f'optimizer_{step}.pt')


class SPG():
    def __init__(self, game):
        self.state = game.get_initial_state()
        self.memory = []
        self.root = None
        self.node = None


In [129]:
gomoku = Gomoku(size=5, win_len=3)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ResNet(gomoku, 4, 64, device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

args = {
    'cpuct': 2,
    'num_simulations': 60,
    'num_iterations': 3,
    'num_self_play': 10,    
    'num_training_steps': 4,
    'num_parallel_games': 4,
    'batch_size': 64,
    'temperature': 1.25,
    'dirichlet_eps': 0.25,
    'dirichlet_alpha': 0.3  # maybe change this
}

az = AlphaZeroParallel(gomoku, model, optimizer, **args)
az.learn()

100%|██████████| 4/4 [00:00<00:00, 12.24it/s]


In [ ]:
gomoku = Gomoku(size=5, win_len=3)
player = 1

args = {
    'cpuct': 2,
    'num_simulations': 600,
    'num_iterations': 8,
    'num_self_play': 500,
    'num_parallel_games': 100,    
    'num_training_steps': 4,
    'batch_size': 128,
    'temperature': 1.25,
    'dirichlet_eps': 0.25,
    'dirichlet_alpha': 0.3  # maybe change this
}

model = ResNet(gomoku, 4, 64, device=device)
state_dict = torch.load('model_2.pt', weights_only=True)
model.load_state_dict(state_dict)
model.eval()

mcts = MCTS(gomoku, model, **args)

state = gomoku.get_initial_state()

while True:
    gomoku.render(state)

    if player == 1:
        valid_actions = gomoku.get_valid_actions(state)
        pretty_valid_actions = [(a // gomoku.size, a % gomoku.size) for a in valid_actions]
        print("valid actions: ", pretty_valid_actions)
        action = input(f"Player {player} action: ")
        if ',' in action:
            action = tuple(map(int, action.split(',')))
            action = action[0] * gomoku.size + action[1]
        else:
            action = int(action)
        
        if action not in valid_actions:
            print("Invalid action")
            continue
        
    else:
        neutral_state = gomoku.change_perspective(state, player)
        mcts_probs = mcts.search(neutral_state)
        action = int(np.argmax(mcts_probs))
        print("MCTS action: ", action)
        print('type', type(action))

    state = gomoku.get_next_state(state, action, player)

    value, done = gomoku.get_value_and_done(state, action)

    if done:
        gomoku.render(state)
        if value == 1:
            print(f"Player {player} wins!")
        else:
            print("Draw!")
        break

    player = gomoku.get_opponent(player)



        


    0  1  2  3  4
 0  0  0  0  0  0
 1  0  0  0  0  0
 2  0  0  0  0  0
 3  0  0  0  0  0
 4  0  0  0  0  0
valid actions:  [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]


    0  1  2  3  4
 0  0  0  0  0  0
 1  0  1  0  0  0
 2  0  0  0  0  0
 3  0  0  0  0  0
 4  0  0  0  0  0
MCTS action:  9
type <class 'int'>
    0  1  2  3  4
 0  0  0  0  0  0
 1  0  1  0  0 -1
 2  0  0  0  0  0
 3  0  0  0  0  0
 4  0  0  0  0  0
valid actions:  [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
    0  1  2  3  4
 0  0  0  0  0  0
 1  0  1  1  0 -1
 2  0  0  0  0  0
 3  0  0  0  0  0
 4  0  0  0  0  0
MCTS action:  24
type <class 'int'>
    0  1  2  3  4
 0  0  0  0  0  0
 1  0  1  1  0 -1
 2  0  0  0  0  0
 3  0  0  0  0  0
 4  0  0  0  0 -1
valid actions:  [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (4, 0), (4, 1), (4, 2), (4, 3)]
    0  1  2  3  4
 0  0  0  0  0  0
 1  0  1  1  0 -1
 2  0  0  1  0  0
 3  0  0  0  0  0
 4  0  0  0  0 -1
MCT